In [9]:
import sys
import os
import numpy as np
import pandas as pd
from typing import List, Optional
from datetime import datetime

base_path = "/home/ec2-user/SageMaker/weakly-supervised-fake-news-classification"

print(base_path)
sys.path.append(base_path)

/home/ec2-user/SageMaker/weakly-supervised-fake-news-classification


In [10]:
!pwd

/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification/src/custom_notebooks


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option("display.max_columns", None)
import pandas as pd

from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_weak_labeled_csv, read_csv, balanced_undersample

In [12]:
weak_labeled_dataset = Datasets.NELA_COVID_COMBINED.weak_labeled_path
train_dataset = Datasets.NELA_ELECTIONS_2020.name
top_n = 0
curated_lfs = True

df = read_weak_labeled_csv(
    weak_labeled_dataset,
    train_dataset,
    top_n=top_n,
    curated_lfs=curated_lfs,
)
print(len(df))

Reading dataset from path: /Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification/data/with_weak_labels/nela/nela-covid-combined-labeled-with-elections-2020-top_0_curated_lfs.csv
Read dataset, dataframe size: 297518
297518


In [13]:
df.head()

,title,content,actual_label,snorkel_majority_vote_label,snorkel_label_model_label,snorkel_label_model_fake_prob,hyper_label_model_label
0,A Postal Worker Filed Three OSHA Complaints ab...,This story was originally published by ProPubl...,0,0,0,0.000080,0
1,Trump: Dr. Fauci Made a Lot of Mistakes Amid C...,President Donald Trump is turning his focus to...,1,1,1,0.926573,1
2,Texas House Speaker says he'll charter a plane...,Texas GOP House Speaker plans to charter a pla...,1,1,1,0.986159,1
3,Indonesia cancels hajj pilgrimage due to coron...,"JAKARTA , Indonesia - Indonesia s government h...",0,0,0,0.000365,0
4,Media Turns On Biden: Why Do You Continue To T...,President Biden said the U.S. has helped evacu...,1,-1,1,0.839016,0


In [20]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
print(accuracy_score(df["actual_label"], df["snorkel_label_model_label"]))

0.8411860794977111


In [23]:
df_size = 100000
largest_fake_probs = df.nlargest(int(df_size/2), "snorkel_label_model_fake_prob")
smallest_fake_probs = df.nsmallest(int(df_size/2), "snorkel_label_model_fake_prob")
combined_df = pd.concat([largest_fake_probs, smallest_fake_probs])

In [24]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
print(accuracy_score(combined_df["actual_label"], combined_df["snorkel_label_model_label"]))

0.9765


In [29]:
weak_labeled_dataset = Datasets.NELA_COVID_COMBINED.weak_labeled_path
train_dataset = Datasets.NELA_COVID_SUBSET.name
top_n = 0
curated_lfs = True

df = read_weak_labeled_csv(
    weak_labeled_dataset,
    train_dataset,
    top_n=top_n,
    curated_lfs=curated_lfs,
)
print(len(df))

Reading dataset from path: /Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification/data/with_weak_labels/nela/nela-covid-combined-labeled-with-covid-subset-top_0_curated_lfs.csv
Read dataset, dataframe size: 297518
297518


In [30]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
print(accuracy_score(df["actual_label"], df["snorkel_label_model_label"]))

0.7856499438689424


In [31]:
df_size = 100000
largest_fake_probs = df.nlargest(int(df_size/2), "snorkel_label_model_fake_prob")
smallest_fake_probs = df.nsmallest(int(df_size/2), "snorkel_label_model_fake_prob")
combined_df = pd.concat([largest_fake_probs, smallest_fake_probs])

In [32]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
print(accuracy_score(combined_df["actual_label"], combined_df["snorkel_label_model_label"]))

0.92295


In [8]:
combined_df["snorkel_label_model_label"].value_counts()

1    50000
0    50000
Name: snorkel_label_model_label, dtype: int64

In [9]:
df = combined_df[["title", "content", "actual_label", "snorkel_label_model_label"]].copy()
df = df.rename(columns={"snorkel_label_model_label": 'weak_label', "content": "text"})
df = df.sample(frac=1)

In [10]:
df.reset_index(drop=True)

,title,text,actual_label,weak_label
0,Masters golf tournament to take place without ...,Aug. 12 ( UPI ) -- Augusta National Golf Club ...,0,0
1,A guide to all the ways the House spending bil...,WASHINGTON - House Democrats on Friday morning...,0,0
2,De Blasio Tweet Singles Out Defiant Jews For P...,Where are all those timid souls who claimed to...,1,1
3,"If anyone can save Joe Biden, its Donald Trump",Is the US ready for Donald Trump s second term...,0,0
4,"Tunisia's president fires prime minister, free...",July 26 ( UPI ) -- Tunisia 's president has fi...,0,0
...,...,...,...,...
99995,Metal Legend Dave Mustaine Calls Out Mask Tyranny,"Dave Mustaine , the guitarist and vocalist of ...",1,1
99996,MSNBC Journalist Claims Calling Coronavirus Wu...,Leftists fear that President Trump may exploit...,1,1
99997,Biden offers sharp attack on Trump as a dark f...,"He framed the election as a stark choice , des...",0,0
99998,"Angels Shohei Ohtani strains arm, wont throw f...","ANAHEIM , Calif. ( AP ) Shohei Ohtani has a ri...",0,0


In [11]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
print(accuracy_score(df["actual_label"], df["weak_label"]))

0.94322


In [12]:
df["text"] = df["title"] + ". " + df["text"]

In [13]:
df.drop(["title"], axis=1, inplace=True)

In [14]:
df.head()

,text,actual_label,weak_label
108047,Masters golf tournament to take place without ...,0,0
13793,A guide to all the ways the House spending bil...,0,0
133962,De Blasio Tweet Singles Out Defiant Jews For P...,1,1
146696,"If anyone can save Joe Biden, its Donald Trump...",0,0
62263,"Tunisia's president fires prime minister, free...",0,0


In [15]:
len(df)

100000

# Add Label Column and create concat dataframe

In [16]:
train_df, eval_df = train_test_split(df, test_size=0.2, stratify=df['weak_label'], random_state=42)
train_df, test_df = train_test_split(train_df, test_size=0.2, stratify=train_df['weak_label'], random_state=42)


In [17]:
print(len(train_df))
print(len(eval_df))
print(len(test_df))

64000
20000
16000


In [18]:
# train_df = train_df.rename(columns={"weak_label": 'label'})
# train_df.drop(["actual_label"], axis=1, inplace=True)
# train_df.head()

In [19]:
train_df_with_actual_labels = train_df.copy()
train_df_with_actual_labels.rename(columns={"actual_label": 'labels'}, inplace=True)
train_df_with_actual_labels.drop(["weak_label"], axis=1, inplace=True)
train_df_with_actual_labels.head()

,text,labels
40433,'Our patience is wearing thin': Biden tells em...,0
132032,"Gold Sets Its Sights on $1,800. This week , Yo...",1
46031,Morning Brief: Canada and U.S. leaning towards...,0
131690,Huge! Japan Recalls 1.63 Million Doses Of Mode...,1
2192,How Ukraine Lost Its Investment Paradise. Afte...,0


In [20]:
train_df_with_weak_labels = train_df.copy()
train_df_with_weak_labels.rename(columns={"weak_label": 'labels'}, inplace=True)
train_df_with_weak_labels.drop(["actual_label"], axis=1, inplace=True)
train_df_with_weak_labels.head()

,text,labels
40433,'Our patience is wearing thin': Biden tells em...,0
132032,"Gold Sets Its Sights on $1,800. This week , Yo...",1
46031,Morning Brief: Canada and U.S. leaning towards...,0
131690,Huge! Japan Recalls 1.63 Million Doses Of Mode...,1
2192,How Ukraine Lost Its Investment Paradise. Afte...,0


In [21]:
eval_df_with_actual_labels = eval_df.copy()
eval_df_with_actual_labels.rename(columns={"actual_label": 'labels'}, inplace=True)
eval_df_with_actual_labels.drop(["weak_label"], axis=1, inplace=True)
eval_df_with_actual_labels.head()

,text,labels
46564,British Author Warns Our Grandkids Will Still ...,1
11703,Senate candidate took money from corporate int...,0
83828,Ive Struggled With Schizophrenia. The Pandemic...,0
170814,Fear Of The Coronavirus Has Absolutely Destroy...,1
92954,My dad listened to Rush Limbaugh attack gay pe...,0


In [22]:
eval_df_with_weak_labels = eval_df.copy()
eval_df_with_weak_labels.rename(columns={"weak_label": 'labels'}, inplace=True)
eval_df_with_weak_labels.drop(["actual_label"], axis=1, inplace=True)
eval_df_with_weak_labels.head()

,text,labels
46564,British Author Warns Our Grandkids Will Still ...,1
11703,Senate candidate took money from corporate int...,0
83828,Ive Struggled With Schizophrenia. The Pandemic...,0
170814,Fear Of The Coronavirus Has Absolutely Destroy...,1
92954,My dad listened to Rush Limbaugh attack gay pe...,0


In [23]:
test_df_with_actual_labels = test_df.copy()
test_df_with_actual_labels.rename(columns={"actual_label": 'labels'}, inplace=True)
test_df_with_actual_labels.drop(["weak_label"], axis=1, inplace=True)
test_df_with_actual_labels.head()

,text,labels
118327,New Couch to Financial Fitness course aims to ...,0
95151,Top Eight. Your Friday night briefing from the...,0
54907,"Priest shot outside French church, suspect arr...",0
48129,Newcastle United 0-2 Manchester City: Raheem S...,0
71623,"Against all odds, Biden transition team is sta...",0


In [24]:
test_df_with_weak_labels = test_df.copy()
test_df_with_weak_labels.rename(columns={"weak_label": 'labels'}, inplace=True)
test_df_with_weak_labels.drop(["actual_label"], axis=1, inplace=True)
test_df_with_weak_labels.head()

,text,labels
118327,New Couch to Financial Fitness course aims to ...,0
95151,Top Eight. Your Friday night briefing from the...,0
54907,"Priest shot outside French church, suspect arr...",1
48129,Newcastle United 0-2 Manchester City: Raheem S...,0
71623,"Against all odds, Biden transition team is sta...",0


In [25]:
import wandb

wandb.login()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: sakdag. Use `wandb login --relogin` to force relogin


True

In [26]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from simpletransformers.classification import ClassificationModel, ClassificationArgs

In [27]:
# import torch

# is_cuda = torch.cuda.is_available()
# if is_cuda:
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")
    
# print(device)

In [28]:
# model_args.eval_batch_size = 32
# model_args.evaluate_during_training = True
# model_args.evaluate_during_training_silent = False
# model_args.evaluate_during_training_steps = 1000
# model_args.manual_seed = 4
# model_args.max_seq_length = 256
# model_args.multiprocessing_chunksize = 5000
# model_args.no_cache = True
# model_args.overwrite_output_dir = True
# model_args.reprocess_input_data = True
# model_args.train_batch_size = 32
# model_args.gradient_accumulation_steps = 2
# model_args.train_custom_parameters_only = False
# model_args.num_train_epochs = 10
# model_args.learning_rate = 0.00000717
model_args = ClassificationArgs(
    num_train_epochs=3, 
    overwrite_output_dir=True, 
    manual_seed = 4,
    use_multiprocessing=False,
    use_multiprocessing_for_evaluation=False,
    learning_rate=0.00000717,
    train_custom_parameters_only = False,
    eval_batch_size = 32,
    evaluate_during_training = False,
    evaluate_during_training_silent = False,
    evaluate_during_training_steps = 1000,
    max_seq_length = 256,
    no_cache = True,
    wandb_project="weakly-supervised-roberta-text-classification",
)
model = ClassificationModel(
    model_type='roberta', 
    model_name='roberta-base', 
    use_cuda=True, 
    num_labels=2, 
    args=model_args
)

Some weights of the model checkpoint at roberta-base were not used when initializing RobertaForSequenceClassification: ['lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should pr

In [ ]:
# Train with weak labels
# model.train_model(train_df_with_weak_labels, eval_df=eval_df_with_weak_labels)
model.train_model(train_df_with_weak_labels, eval_df=eval_df_with_weak_labels)

Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Running Epoch 0 of 3:   0%|          | 0/8000 [00:00<?, ?it/s]

In [ ]:
# Train with actual labels
# model.train_model(train_df_with_actual_labels, eval_df=eval_df_with_actual_labels)

In [ ]:
now = int(round(datetime.now().timestamp() * 1000))
print(f"weak_labeled_dataset: {weak_labeled_dataset}")
print(f"train_dataset: {train_dataset}")
print(f"top_n: {top_n}")

In [ ]:
print(f"Middle eliminated, {df_size} data size, snorkel label model")

In [ ]:
test_predictions, raw_outputs = model.predict(list(test_df_with_weak_labels['text']))

In [ ]:
cr = classification_report(test_df_with_weak_labels['labels'], test_predictions, digits=5, output_dict=True)
# print(cr)
cr_df = pd.DataFrame(cr).transpose()
print(cr_df)
# cr_df.to_csv(f"/home/ec2-user/SageMaker/temp/elections_covid_top10_3epoch/test_with_weak_labels_{now}.csv")

In [ ]:
test_predictions, raw_outputs = model.predict(list(test_df_with_actual_labels['text']))

In [ ]:
cr = classification_report(test_df_with_actual_labels['labels'], test_predictions, digits=5, output_dict=True)
# print(cr)
cr_df = pd.DataFrame(cr).transpose()
print(cr_df)
# cr_df.to_csv(f"/home/ec2-user/SageMaker/temp/elections_covid_top10_3epoch/test_with_actual_labels_{now}.csv")

In [ ]:
manual_test_df = pd.read_csv("/home/ec2-user/SageMaker/temp/manually-factchecked-piiingz.csv")

In [ ]:
manual_test_df = manual_test_df[["title", "content", "label"]].copy()
manual_test_df["text"] = manual_test_df["title"] + ". " + manual_test_df["content"]
manual_test_df = manual_test_df[["text", "label"]].copy()

In [ ]:
test_predictions, raw_outputs = model.predict(list(manual_test_df['text']))

In [ ]:
cr = classification_report(manual_test_df['label'], test_predictions, digits=5, output_dict=True)
# print(cr)
cr_df = pd.DataFrame(cr).transpose()
print(cr_df)
# cr_df.to_csv(f"/home/ec2-user/SageMaker/temp/elections_covid_top10_3epoch/test_with_manual_set_{now}.csv")


In [ ]:
wandb.finish()